# Demo DiCaprio · CUPID + MOTIF su Kaggle

Notebook di prova per la **Fase 1** del progetto M1: installa CUPID e MOTIF dai repo ufficiali `polimi-ispl`, prepara le clip della demo DiCaprio (8 riferimenti, 3 video veri, 4 deepfake; le stesse per entrambi i metodi) e confronta i risultati con quelli pubblicati.

| Metodo | Risultato pubblicato |
|---|---|
| CUPID | AUC 1.000, margine **+0.011314** (15 frame, risoluzione nativa) |
| MOTIF | AUC 1.000, margine **+3.273773** (batch size 8) |

## Impostazioni Kaggle (pannello a destra, *Settings*)
1. **Accelerator → GPU T4 x2.** Evitare la P100: le build recenti di PyTorch non la supportano più bene.
2. **Internet → On** (richiede account verificato con telefono).
3. Poi *Run All*. La prima esecuzione richiede circa 15–25 minuti (installazione, compilazione di `nvdiffrast`, download dei pesi).

## Video sorgente (YouTube + TikTok)
Da Kaggle YouTube spesso blocca i download anonimi ("Sign in to confirm you're not a bot"). Il notebook prova in quest'ordine:
1. **Clip già pronte** in un dataset Kaggle allegato (file `ref_1.mp4` … `test_fake_4.mp4`): vengono usate così come sono.
2. **Video sorgente** in un dataset allegato (nomi che contengono l'ID, es. `... [usqUrdA_R3I].mp4`): vengono tagliati con `prepare_clips.sh`.
3. **Download con yt-dlp**, usando un eventuale `cookies.txt` trovato tra gli input.

**Se YouTube blocca il download:** sul PC, con un browser loggato su YouTube, esporta i cookie in formato Netscape (es. estensione *Get cookies.txt LOCALLY*), salvali come `cookies.txt`, caricali come **dataset Kaggle privato** e aggiungilo al notebook (*Add Input*). I cookie danno accesso all'account: meglio un account secondario, dataset privato, da cancellare dopo l'uso.

Le clip della demo restano soggette ai diritti dei proprietari originali: non pubblicare dataset o output che le contengono.

In [ ]:
# 1 · Configurazione, helper e controllo della GPU
import glob
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
INPUTS = Path("/kaggle/input")
REPOS = WORK / "repos"
DEMO = WORK / "demo"
DOWNLOADS = DEMO / "downloads"
CLIPS = DEMO / "clips"
RESULTS = DEMO / "results"
for path in (REPOS, DOWNLOADS, CLIPS, RESULTS):
    path.mkdir(parents=True, exist_ok=True)

PY = sys.executable


def run(cmd, check=True, cwd=None):
    """Esegue un comando bash mostrando l'output riga per riga nella cella."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, executable="/bin/bash", cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"comando fallito (exit {code}): {cmd}")
    return code


def out(cmd):
    """Esegue un comando e ne restituisce lo stdout (stringa vuota se fallisce)."""
    return subprocess.run(cmd, shell=True, executable="/bin/bash",
                          capture_output=True, text=True).stdout.strip()


gpu = out("nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader")
if not gpu:
    raise RuntimeError("Nessuna GPU NVIDIA: Settings → Accelerator → GPU T4 x2, poi riavvia la sessione.")
print("GPU:", gpu)
compute_cap = gpu.splitlines()[0].split(",")[1].strip()
if float(compute_cap) < 7.0:
    print(f"ATTENZIONE: compute capability {compute_cap} (P100?). Passa a GPU T4 x2 se l'installazione fallisce.")
# Compila nvdiffrast solo per l'architettura presente: molto più veloce.
os.environ["TORCH_CUDA_ARCH_LIST"] = compute_cap

cuda_candidates = ["/usr/local/cuda", *sorted(glob.glob("/usr/local/cuda-*"), reverse=True)]
cuda_home = next((p for p in cuda_candidates if Path(p, "bin", "nvcc").exists()), None)
if cuda_home is None:
    raise RuntimeError("nvcc non trovato: serve il CUDA toolkit per compilare nvdiffrast (CUPID).")
os.environ["CUDA_HOME"] = cuda_home
os.environ["PATH"] = f"{cuda_home}/bin:" + os.environ["PATH"]
match = re.search(r"release (\d+)\.(\d+)", out("nvcc --version"))
NVCC = (int(match[1]), int(match[2]))
print(f"nvcc {NVCC[0]}.{NVCC[1]} in {cuda_home}")
print("Python", sys.version.split()[0])

# nvdiffrast 0.3.3 (richiesto da CUPID) compila il plugin CUDA al primo uso e poi fa
# `import nvdiffrast_plugin`; con torch recenti la cartella di build non è nel path e
# l'import fallisce (ModuleNotFoundError). Fissiamo la cartella, la creiamo subito
# (Python ignora per sempre le cartelle di PYTHONPATH assenti all'avvio) e la mettiamo nel path.
TORCH_EXT = WORK / "torch_extensions"
PLUGIN_DIR = TORCH_EXT / "nvdiffrast_plugin"
PLUGIN_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_EXTENSIONS_DIR"] = str(TORCH_EXT)
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(PLUGIN_DIR), os.environ.get("PYTHONPATH")]))

if not out("curl -sI --max-time 10 https://huggingface.co | head -1"):
    raise RuntimeError("Internet disattivato: Settings → Internet → On.")
print("Internet OK")

In [ ]:
# 2 · FFmpeg (librerie condivise per torchcodec, eseguibile con libx264) e font DejaVu
need = []
if not shutil.which("ffmpeg") or "libx264" not in out("ffmpeg -hide_banner -encoders"):
    need.append("ffmpeg")
if not out("ldconfig -p | grep libavutil"):
    need.append("ffmpeg")  # il pacchetto apt porta con sé le librerie libav* condivise
if not Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf").exists():
    need.append("fonts-dejavu-core")
if need:
    run("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq " + " ".join(sorted(set(need))))
print(out("ffmpeg -version | head -1"))
print("libx264:", "libx264" in out("ffmpeg -hide_banner -encoders"))

In [ ]:
# 3 · PyTorch con la stessa versione CUDA *major* di nvcc (necessario per compilare nvdiffrast)
if NVCC[0] >= 13:
    TORCH, TORCHVISION, TORCHCODEC, INDEX = "2.9.0", "0.24.0", "0.8.0", "cu130"
else:
    TORCH, TORCHVISION, TORCHCODEC = "2.8.0", "0.23.0", "0.7.0"
    INDEX = "cu128" if NVCC >= (12, 8) else "cu126"

current = out(f'{PY} -c "import torch; print(torch.__version__)"')
print("torch installato:", current or "nessuno", "→ richiesto:", f"{TORCH}+{INDEX}")
if current != f"{TORCH}+{INDEX}":
    run(f"{PY} -m pip install -q torch=={TORCH} torchvision=={TORCHVISION} "
        f"--index-url https://download.pytorch.org/whl/{INDEX}")
run(f"{PY} -m pip install -q torchcodec=={TORCHCODEC}")

# Vincoli per impedire che i passi successivi cambino torch.
CONSTRAINTS = WORK / "constraints.txt"
CONSTRAINTS.write_text(f"torch=={TORCH}\ntorchvision=={TORCHVISION}\ntorchcodec=={TORCHCODEC}\n")

In [ ]:
# 4 · Clone dei repo ufficiali e installazione (con gli extra della demo)
for name in ("CUPID", "MOTIF"):
    if not (REPOS / name).exists():
        run(f"git clone --depth 1 https://github.com/polimi-ispl/{name}.git {REPOS / name}")
    print(name, out(f"git -C {REPOS / name} log -1 --format='%h %cd' --date=short"))

run(f'{PY} -m pip install -q -c {CONSTRAINTS} -e "{REPOS / "CUPID"}[demo]" -e "{REPOS / "MOTIF"}[demo]" "yt-dlp[default]"')

# yt-dlp ha bisogno di un runtime JavaScript (deno) per YouTube.
if not shutil.which("deno"):
    run(f"{PY} -m pip install -q deno", check=False)
if not shutil.which("deno"):
    run("curl -fsSL https://deno.land/install.sh | sh -s -- -y", check=False)
    os.environ["PATH"] = f"{Path.home() / '.deno' / 'bin'}:" + os.environ["PATH"]
print("deno:", shutil.which("deno") or "non trovato (YouTube potrebbe fallire)")

In [ ]:
# 5 · Verifica dell'ambiente e prima compilazione di nvdiffrast (qualche minuto, poi è in cache)
check = WORK / "check_env.py"
check.write_text("""
import torch, torchcodec
import nvdiffrast.torch as dr
import cupid, motif
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))
print("torchcodec", torchcodec.__version__)
ctx = dr.RasterizeCudaContext(device="cuda:0")
print("nvdiffrast compilato e funzionante")
from torchcodec.decoders import VideoDecoder
print("torchcodec carica FFmpeg correttamente")
""")
if run(f"{PY} {check}", check=False) != 0:
    # Al primo avvio la compilazione può riuscire ma l'import nello stesso processo no:
    # in un processo nuovo il plugin è già compilato e viene solo caricato.
    print("\nPrimo tentativo fallito, riprovo con il plugin già compilato:")
    print("file nella cartella del plugin:", sorted(p.name for p in PLUGIN_DIR.iterdir()))
    # Se il .so è finito altrove (es. ~/.cache/torch_extensions), aggiungiamo quella cartella al path.
    built = sorted({p.parent for root in (TORCH_EXT, Path.home() / ".cache" / "torch_extensions")
                    if root.exists() for p in root.rglob("nvdiffrast_plugin*.so")})
    print("plugin compilati trovati:", [str(p) for p in built] or "nessuno")
    if built:
        os.environ["PYTHONPATH"] = os.pathsep.join([*map(str, built), os.environ["PYTHONPATH"]])
    run(f"{PY} {check}")

In [ ]:
# 6 · Video della demo: clip pronte → sorgenti allegate → download con yt-dlp
CLIP_NAMES = ([f"ref_{i}.mp4" for i in range(1, 9)]
              + [f"test_real_{i}.mp4" for i in range(1, 4)]
              + [f"test_fake_{i}.mp4" for i in range(1, 5)])
links_file = REPOS / "CUPID" / "demo" / "links.txt"
URLS = [line.strip() for line in links_file.read_text().splitlines()
        if line.strip().startswith("http")]
VIDEO_IDS = {url: re.search(r"(?:v=|/video/)([\w-]+)", url)[1] for url in URLS}

input_videos = list(INPUTS.rglob("*.mp4")) if INPUTS.exists() else []

# 1) Clip già pronte negli input?
ready = {f.name: f for f in input_videos if f.name in CLIP_NAMES}
CLIPS_READY = len(ready) == len(CLIP_NAMES)
if CLIPS_READY:
    for name, src in ready.items():
        shutil.copy2(src, CLIPS / name)
    print(f"Uso le {len(ready)} clip già pronte trovate in {INPUTS}")
else:
    # 2) Video sorgente negli input?
    missing = []
    for url, vid in VIDEO_IDS.items():
        if any(vid in f.name for f in DOWNLOADS.glob("*.mp4")):
            continue
        match = next((f for f in input_videos if vid in f.name), None)
        if match:
            shutil.copy2(match, DOWNLOADS / match.name)
            print("sorgente dagli input:", match.name)
        else:
            missing.append(url)

    # 3) Download dei mancanti
    cookies_src = next(INPUTS.rglob("cookies*.txt"), None) if INPUTS.exists() else None
    cookie_arg = ""
    if cookies_src:
        # Copia scrivibile: yt-dlp aggiorna il file e /kaggle/input è in sola lettura.
        shutil.copy2(cookies_src, WORK / "cookies.txt")
        cookie_arg = f"--cookies {WORK / 'cookies.txt'}"
        print("Uso i cookie di", cookies_src)
    failed = []
    for url in missing:
        code = run(
            f'yt-dlp {cookie_arg} --no-overwrites --continue --restrict-filenames '
            f'--format "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b" --merge-output-format mp4 '
            f'--output "{DOWNLOADS}/%(uploader)s - %(title)s [%(id)s].%(ext)s" "{url}"',
            check=False,
        )
        if code != 0:
            failed.append(url)
    if failed:
        print("\nDownload falliti:")
        print("\n".join("  " + u for u in failed))
        raise RuntimeError("Alcuni video non sono stati scaricati: vedi la sezione 'Video sorgente' in cima "
                           "(cookies.txt o video caricati come dataset Kaggle).")
    print("Tutti i video sorgente sono disponibili.")

In [ ]:
# 7 · Taglio delle clip con lo script ufficiale (configurazione identica in CUPID e MOTIF)
if not CLIPS_READY:
    run(f"INPUT_DIR={DOWNLOADS} OUTPUT_DIR={CLIPS} bash prepare_clips.sh",
        cwd=REPOS / "CUPID" / "demo")
present = sorted(p.name for p in CLIPS.glob("*.mp4"))
assert all(name in present for name in CLIP_NAMES), f"clip mancanti: {set(CLIP_NAMES) - set(present)}"
for name in CLIP_NAMES:
    info = out(f"ffprobe -v error -select_streams v:0 -count_frames "
               f"-show_entries stream=width,height,r_frame_rate,nb_read_frames -of csv=p=0 {CLIPS / name}")
    print(f"{name:<18} {info}")

In [ ]:
# 8 · Demo CUPID (15 frame equispaziati, risoluzione nativa) + video di confronto
cupid_out = RESULTS / "cupid"
run(f"{PY} quickstart.py --clips {CLIPS} --out {cupid_out} --device cuda:0",
    cwd=REPOS / "CUPID" / "demo", check=False)

cupid_scores = json.loads((cupid_out / "scores.json").read_text())
recorded = json.loads((REPOS / "CUPID" / "docs" / "demo-results.json").read_text())
rec_by_clip = {r["clip"]: r["score"] for r in recorded["results"]}
print(f"\n{'clip':<18}{'label':<6}{'nostro':>10}{'pubblicato':>12}{'diff':>10}")
for r in sorted(cupid_scores["results"], key=lambda r: r["clip"]):
    ref = rec_by_clip.get(r["clip"])
    ref_s = "-" if ref is None else f"{ref:.4f}"
    diff = "-" if ref is None else f"{r['score'] - ref:+.4f}"
    print(f"{r['clip']:<18}{r['label']:<6}{r['score']:>10.4f}{ref_s:>12}{diff:>10}")
print(f"\nAUC    nostro {cupid_scores['auc']:.3f}  | pubblicato {recorded['auc']:.3f}")
print(f"margin nostro {cupid_scores['margin']:+.6f} | pubblicato {recorded['margin']:+.6f}")

In [ ]:
# 8b · Mappa di interpretabilità CUPID: Real 2 vs Fake 2
from IPython.display import Image, display
gif = cupid_out / "comparison.gif"
if gif.exists():
    display(Image(filename=str(gif)))
else:
    print("comparison.gif non generata: guarda l'output della cella 8 (gli score restano validi).")

In [ ]:
# 9 · Demo MOTIF (tutti i frame, finestre da 75 frame, batch size 8 come il risultato pubblicato)
motif_out = RESULTS / "motif"
run(f"{PY} quickstart.py --clips {CLIPS} --out {motif_out} --device cuda:0 --batch-size 8",
    cwd=REPOS / "MOTIF" / "demo")

motif_scores = json.loads((motif_out / "scores.json").read_text())
print(f"\n{'clip':<18}{'label':<6}{'fused':>10}{'global':>10}{'mouth':>10}")
for r in sorted(motif_scores["results"], key=lambda r: r["clip"]):
    print(f"{r['clip']:<18}{r['label']:<6}{r['score']:>10.4f}{r['global_score']:>10.4f}{r['mouth_score']:>10.4f}")
print(f"\nAUC    nostro {motif_scores['auc']:.3f}  | pubblicato 1.000")
print(f"margin nostro {motif_scores['margin']:+.6f} | pubblicato +3.273773")

In [ ]:
# 10 · Riepilogo e archivio dei risultati (scaricabile dal pannello Output)
summary = {
    "gpu": gpu,
    "torch": f"{TORCH}+{INDEX}",
    "nvcc": f"{NVCC[0]}.{NVCC[1]}",
    "cupid": {"auc": cupid_scores["auc"], "margin": cupid_scores["margin"],
              "published": {"auc": 1.0, "margin": 0.011314}},
    "motif": {"auc": motif_scores["auc"], "margin": motif_scores["margin"],
              "published": {"auc": 1.0, "margin": 3.273773}},
    "clip_info": {name: out(f"ffprobe -v error -select_streams v:0 -show_entries "
                            f"stream=width,height,r_frame_rate -of csv=p=0 {CLIPS / name}")
                  for name in CLIP_NAMES},
}
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps({k: summary[k] for k in ("gpu", "torch", "cupid", "motif")}, indent=2))

for name, s in (("CUPID", cupid_scores), ("MOTIF", motif_scores)):
    status = "OK" if s["auc"] == 1.0 and s["margin"] > 0 else "DA CONTROLLARE"
    print(f"{name}: {status}")

# Solo score e mappe: niente clip video nell'archivio.
archive = shutil.make_archive(str(WORK / "demo_results"), "zip", RESULTS)
print("Archivio:", archive)

## Note per leggere i risultati
- **Piccole differenze negli score sono normali.** La risoluzione scaricata da yt-dlp, la versione di FFmpeg e la GPU possono cambiare di poco i valori. Conta che l'**AUC sia 1.000** e che il **margine sia positivo** e dello stesso ordine di grandezza di quello pubblicato.
- **MOTIF dipende dal batch size.** L'estrazione dei coefficienti cambia leggermente con il batch, per questo si usa 8, come nel risultato pubblicato.
- **Gli score non sono probabilità.** CUPID restituisce una cosine similarity massima, MOTIF uno z-score calibrato per POI: le due scale non sono confrontabili tra loro.
- **Tempi da annotare per la pianificazione:** quanto impiega MOTIF per frame (decodifica tutti i frame) rispetto a CUPID (15 frame per video). Servono per stimare il costo su 15–20 POI.